In [89]:
import jax
import jax.numpy as jnp
import jaxls
import matplotlib.cm as cm

In [90]:
# Dynamics / observation definitions.
m = 1.0
dt = 0.1

A = jnp.block([[jnp.eye(3), dt * jnp.eye(3)], [jnp.zeros((3, 3)), jnp.eye(3)]])
B = jnp.concatenate([jnp.zeros((3, 3)), (dt / m) * jnp.eye(3)], axis=0)
C = jnp.concatenate([jnp.eye(3), jnp.zeros((3, 3))], axis=1)

Q = dt * 0.1 * jnp.eye(6)
R = dt * 2.0 * jnp.eye(3)

def f(x, u):
    return A @ x + B @ u

def h(x):
    return C @ x

# Simulate trajectory
horizon = 50
scale= 3.0

t = jnp.linspace(0, 2 * jnp.pi, horizon + 1)

# Lemniscate: x = sin(t), y = sin(2t)/2, z = 0
pos_ref = scale * jnp.stack([
    jnp.sin(t),
    jnp.sin(2 * t) / 2,
    jnp.ones_like(t),
], axis=1)

# Finite-difference velocities
vel_ref = jnp.diff(pos_ref, axis=0) / dt
vel_ref = jnp.concatenate([vel_ref, vel_ref[-1:]], axis=0)  # repeat last

X_ref = jnp.concatenate([pos_ref, vel_ref], axis=1)


# Initial state
mu0 = X_ref[0]
Sigma0 = 2.0 * jnp.eye(6) 
x0 = jax.random.multivariate_normal(jax.random.PRNGKey(0), mu0, Sigma0)

# Simulate with noise
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)
W = jax.random.multivariate_normal(k1, jnp.zeros(6), Q, (horizon,))
V = jax.random.multivariate_normal(k2, jnp.zeros(3), R, (horizon,))

X = jnp.zeros((horizon + 1, 6))
Y = jnp.zeros((horizon + 1, 3))

# PD gains
Kp = 10.0 * jnp.eye(3)
Kd = 5.0 * jnp.eye(3)

# Simulate with PD tracking
x = x0
X = X.at[0].set(x)
Y = Y.at[0].set(jnp.inf * jnp.ones(3))

kf_means = jnp.zeros((horizon + 1, 6))
kf_Sigmas = jnp.zeros((horizon + 1, 6, 6))

mu = mu0
Sigma = Sigma0
kf_means = kf_means.at[0].set(mu)
kf_Sigmas = kf_Sigmas.at[0].set(Sigma)

for t_idx in range(horizon):
    pos = x[:3]
    vel = x[3:]
    pos_des = pos_ref[t_idx + 1]
    vel_des = vel_ref[t_idx + 1]

    u = Kp @ (pos_des - pos) + Kd @ (vel_des - vel)
    U = U.at[t_idx].set(u)

    x = f(x, u) + W[t_idx]
    y = h(x) + V[t_idx]

    mu_bar = f(mu, u)
    Sigma_bar = A @ Sigma @ A.T + Q
    S = C @ Sigma_bar @ C.T + R
    K = Sigma_bar @ C.T @ jnp.linalg.inv(S)
    mu = mu_bar + K @ (y - C @ mu_bar)
    Sigma = (jnp.eye(6) - K @ C) @ Sigma_bar

    kf_means = kf_means.at[t_idx + 1].set(mu)
    kf_Sigmas = kf_Sigmas.at[t_idx + 1].set(Sigma)

    X = X.at[t_idx + 1].set(x)
    Y = Y.at[t_idx + 1].set(y)

In [91]:
class StateVar(jaxls.Var[jax.Array], default_factory=lambda: jnp.zeros(6)):
    """Drone state variable, consisting of position and velocity."""

drone_state_vars = StateVar(id=jnp.arange(horizon+1))

@jaxls.Cost.factory
def dynamics_cost(
    vals: jaxls.VarValues,
    state_curr: StateVar,
    state_next: StateVar,
    u: jax.Array,
) -> jax.Array:
    """Cost enforcing the dynamics constraint between consecutive states."""
    x_curr = vals[state_curr]
    x_next = vals[state_next]

    # Dynamics residual weighted by process noise covariance sqrt.
    return jnp.real(jax.scipy.linalg.sqrtm(jnp.linalg.inv(Q))) @ (x_next - f(x_curr, u))

@jaxls.Cost.factory
def measurement_cost(
    vals: jaxls.VarValues,
    state: StateVar,
    y: jax.Array,
) -> jax.Array:
    """Cost enforcing the measurement constraint between state and observation."""
    x = vals[state]

    # Measurement residual weighted by measurement noise covariance sqrt.
    return jnp.real(jax.scipy.linalg.sqrtm(jnp.linalg.inv(R))) @ (h(x) - y)

@jaxls.Cost.factory
def prior_cost(
    vals: jaxls.VarValues,
    state: StateVar,
    mean: jax.Array = mu0,
    cov: jax.Array = Sigma0,
) -> jax.Array:
    """Cost enforcing the prior constraint on the initial state."""
    x = vals[state]

    # Prior residual weighted by prior covariance sqrt.
    return jnp.real(jax.scipy.linalg.sqrtm(jnp.linalg.inv(Sigma0))) @ (x - mu0)

costs: list[jaxls.Cost] = [
    prior_cost(StateVar(id=0), mu0, Sigma0),
    dynamics_cost(StateVar(id=jnp.arange(horizon)), StateVar(id=jnp.arange(1, horizon+1)), u=U),
    measurement_cost(StateVar(id=jnp.arange(1, horizon+1)), Y[1:]),
]

initial_guess = jaxls.VarValues.make([drone_state_vars.with_value(0.1 + jnp.tile(mu0, (horizon+1,1)))])
problem = jaxls.LeastSquaresProblem(costs, [drone_state_vars])
problem = problem.analyze()

2026-03-24 09:54:10.352 | INFO     | jaxls._py310._problem:analyze:121 - Building optimization problem with 101 terms and 51 variables: 101 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
2026-03-24 09:54:10.440 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 50 costs, 1 variables each: measurement_cost
2026-03-24 09:54:10.485 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 50 costs, 2 variables each: dynamics_cost
2026-03-24 09:54:10.507 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 1 costs, 1 variables each: prior_cost


In [92]:
solution = problem.solve(initial_guess)
cov = problem.make_covariance_estimator(solution)
covariance_jit = jax.jit(cov.covariance)

2026-03-24 09:54:14.789 | INFO     | jaxls._py310.utils:_log:23 -  step #0: cost=10541.3096 lambd=0.0005 inexact_tol=1.0e-02
2026-03-24 09:54:14.789 | INFO     | jaxls._py310.utils:_log:23 -      - measurement_cost(50): 2669.64355 (avg 17.79762)
2026-03-24 09:54:14.790 | INFO     | jaxls._py310.utils:_log:23 -      - dynamics_cost(50): 7871.63623 (avg 26.23879)
2026-03-24 09:54:14.790 | INFO     | jaxls._py310.utils:_log:23 -      - prior_cost(1):  0.03000 (avg 0.00500)
2026-03-24 09:54:14.794 | INFO     | jaxls._py310.utils:_log:23 -      accepted=True ATb_norm=5.25e+02 cost_prev=10541.3115 cost_new=135.7768
2026-03-24 09:54:14.794 | INFO     | jaxls._py310.utils:_log:23 -  step #1: cost=135.7767 lambd=0.0003 inexact_tol=1.0e-02
2026-03-24 09:54:14.795 | INFO     | jaxls._py310.utils:_log:23 -      - measurement_cost(50): 117.33408 (avg 0.78223)
2026-03-24 09:54:14.795 | INFO     | jaxls._py310.utils:_log:23 -      - dynamics_cost(50): 11.44015 (avg 0.03813)
2026-03-24 09:54:14.795 | 

In [93]:
import numpy as onp
import trimesh
import viser
import time

# ── Extract results ──────────────────────────────────────────────
X_smooth = onp.array(jnp.stack([solution[StateVar(id=t)] for t in range(horizon + 1)]))
pos_smooth = X_smooth[:, :3]
pos_gt = onp.array(X[:, :3])
pos_meas = onp.array(Y[1:])

# ── Viser setup ──────────────────────────────────────────────────
if 'server' not in dir() or server is None:
    server = viser.ViserServer()
else:
    server.scene.reset()
    server.gui.reset()

alphas = onp.linspace(0.2, 1.0, horizon + 1)

server.scene.add_grid("grid")
# Ground truth trajectory (green)
gt_colors = cm.Greens(alphas)[:, :3]
gt_line_colors = onp.stack([gt_colors[:-1], gt_colors[1:]], axis=1)
gt_segments = onp.stack([pos_gt[:-1], pos_gt[1:]], axis=1)
gt_trajectory = server.scene.add_line_segments("gt_trajectory", gt_segments, colors=gt_line_colors)
gt_points = server.scene.add_point_cloud(
    "gt_points", pos_gt,
    colors=gt_colors, point_size=0.05, point_shape="circle",
)

# Measurements (red)
meas_colors = cm.Reds(alphas)[1:, :3]
measurements = server.scene.add_point_cloud(
    "measurements", pos_meas,
    colors=meas_colors, point_size=0.07, point_shape="diamond",
)

# Smoothed trajectory (blue)
smooth_colors = cm.Blues(alphas)[:, :3]
smooth_line_colors = onp.stack([smooth_colors[:-1], smooth_colors[1:]], axis=1)
smooth_segments = onp.stack([pos_smooth[:-1], pos_smooth[1:]], axis=1)
server.scene.add_line_segments("smooth_trajectory", smooth_segments, colors=smooth_line_colors)
server.scene.add_point_cloud(
    "smooth_points", pos_smooth,
    colors=smooth_colors, point_size=0.05, point_shape="circle",
)

# ── Covariance ellipsoids ────────────────────────────────────────
unit_sphere = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
n_sigma = 2.0  # 2-sigma ellipsoids

covariances = []

for t in range(horizon + 1):
    # Marginal covariance for state t → extract 3x3 position block.
    cov_full = onp.array(covariance_jit(StateVar(id=t)))  # (6, 6)
    cov_pos = cov_full[:3, :3]

    # Eigen-decomposition → ellipsoid axes.
    eigvals, eigvecs = onp.linalg.eigh(cov_pos)
    radii = n_sigma * onp.sqrt(onp.maximum(eigvals, 1e-10))

    # Build 4×4 affine: rotation * scale + translation.
    transform = onp.eye(4)
    transform[:3, :3] = eigvecs @ onp.diag(radii)
    transform[:3, 3] = pos_smooth[t]

    ellipsoid = unit_sphere.copy().apply_transform(transform)

    # Translucent blue ellipsoid.
    n_verts = len(ellipsoid.vertices)
    rgba = onp.tile([50, 100, 255, 60], (n_verts, 1)).astype(onp.uint8)

    covariances.append(server.scene.add_mesh_simple(
        f"cov_{t}",
        vertices=onp.array(ellipsoid.vertices, dtype=onp.float32),
        faces=onp.array(ellipsoid.faces, dtype=onp.uint32),
        color=(0.2, 0.4, 1.0),
        opacity=0.15,
    ))

# ── Kalman filter trajectory (orange) ────────────────────────────
pos_kf = onp.array(kf_means[:, :3])
kf_colors = cm.Oranges(alphas)[:, :3]
kf_line_colors = onp.stack([kf_colors[:-1], kf_colors[1:]], axis=1)
kf_segments = onp.stack([pos_kf[:-1], pos_kf[1:]], axis=1)
kf_trajectory = server.scene.add_line_segments("kf_trajectory", kf_segments, colors=kf_line_colors)
kf_points = server.scene.add_point_cloud(
    "kf_points", pos_kf,
    colors=kf_colors, point_size=0.05, point_shape="circle",
)

# KF covariance ellipsoids
kf_covariances = []
for t in range(horizon + 1):
    cov_pos = onp.array(kf_Sigmas[t, :3, :3])

    eigvals, eigvecs = onp.linalg.eigh(cov_pos)
    radii = n_sigma * onp.sqrt(onp.maximum(eigvals, 1e-10))

    transform = onp.eye(4)
    transform[:3, :3] = eigvecs @ onp.diag(radii)
    transform[:3, 3] = pos_kf[t]

    ellipsoid = unit_sphere.copy().apply_transform(transform)

    kf_covariances.append(server.scene.add_mesh_simple(
        f"kf_cov_{t}",
        vertices=onp.array(ellipsoid.vertices, dtype=onp.float32),
        faces=onp.array(ellipsoid.faces, dtype=onp.uint32),
        color=(1.0, 0.5, 0.0),
        opacity=0.15,
    ))

# ── GUI: toggle visibility ───────────────────────────────────────
show_gt = server.gui.add_checkbox("Show ground truth", initial_value=True)
show_meas = server.gui.add_checkbox("Show measurements", initial_value=True)
show_cov = server.gui.add_checkbox("Show covariance", initial_value=True)
show_kf = server.gui.add_checkbox("Show Kalman filter", initial_value=True)
show_kf_cov = server.gui.add_checkbox("Show KF covariance", initial_value=True)

@show_kf.on_update
def _(_):
    kf_trajectory.visible = show_kf.value
    kf_points.visible = show_kf.value

@show_kf_cov.on_update
def _(_):
    for t in range(horizon + 1):
        kf_covariances[t].visible = show_kf_cov.value

@show_gt.on_update
def _(_):
    gt_trajectory.visible = show_gt.value
    gt_points.visible = show_gt.value

@show_meas.on_update
def _(_):
    measurements.visible = show_meas.value

@show_cov.on_update
def _(_):
    for t in range(horizon + 1):
        covariances[t].visible = show_cov.value

# ── uPlot: per-dimension comparison ──────────────────────────────
import viser.uplot

timesteps = onp.arange(horizon + 1, dtype=onp.float64) * dt

# Pad measurements to length horizon+1 using prior position at t=0
meas_padded = onp.concatenate([pos_gt[:1, :], pos_meas], axis=0)

dim_labels = ["x", "y", "z"]
with server.gui.add_folder("Plots", expand_by_default=True):
    for d in range(3):
        gt_d = onp.array(pos_gt[:, d], dtype=onp.float64)
        smooth_d = onp.array(pos_smooth[:, d], dtype=onp.float64)
        kf_d = onp.array(pos_kf[:, d], dtype=onp.float64)
        meas_d = onp.array(meas_padded[:, d], dtype=onp.float64)

        # Smoother ±2σ
        smooth_std = onp.array([
            2.0 * onp.sqrt(max(float(covariance_jit(StateVar(id=t))[d, d]), 1e-10))
            for t in range(horizon + 1)
        ])
        smooth_upper = smooth_d + smooth_std
        smooth_lower = smooth_d - smooth_std

        # KF ±2σ
        kf_std = 2.0 * onp.sqrt(onp.maximum(
            onp.array(kf_Sigmas[:, d, d], dtype=onp.float64), 1e-10
        ))
        kf_upper = kf_d + kf_std
        kf_lower = kf_d - kf_std

        server.gui.add_uplot(
            data=(
                timesteps,
                gt_d,
                meas_d,
                smooth_d,
                smooth_upper,
                smooth_lower,
                kf_d,
                kf_upper,
                kf_lower,
            ),
            series=(
                viser.uplot.Series(label="t"),
                viser.uplot.Series(label="GT", stroke="#22aa22", width=2),
                viser.uplot.Series(label="Meas", stroke="#cc3333", width=1),
                viser.uplot.Series(label="Smooth", stroke="#2255dd", width=2),
                viser.uplot.Series(label="+2σ", stroke="#aabbff", width=1),
                viser.uplot.Series(label="−2σ", stroke="#aabbff", width=1),
                viser.uplot.Series(label="KF", stroke="#ee8800", width=2),
                viser.uplot.Series(label="+2σ ", stroke="#ffcc88", width=1),
                viser.uplot.Series(label="−2σ ", stroke="#ffcc88", width=1),
            ),
            title=f"{dim_labels[d]}(t)",
            scales={
                "x": viser.uplot.Scale(time=False, auto=True),
                "y": viser.uplot.Scale(auto=True),
            },
            legend=viser.uplot.Legend(show=False),
            aspect=2.5,
        )

(viser) Connection opened (3, 1 total), 325 persistent messages

(viser) Connection closed (3, 0 total)